# 4. La transparence des développeurs : Stanford et l'AI Act

Deux sources complètent la mesure d'opacité construite à partir d'Epoch (notebook 1) :

- le **Foundation Model Transparency Index** (Stanford) : une centaine d'indicateurs notés 0/1 pour les grands développeurs, en octobre 2023, mai 2024 et décembre 2025 ;
- la liste officielle des **signataires du code de bonnes pratiques** européen, qui accompagne les obligations de l'AI Act en vigueur depuis le 2 août 2025.

**Questions**
1. La transparence des grands développeurs progresse-t-elle ou recule-t-elle entre 2023 et 2025 ?
2. Les deux mesures de l'opacité (Stanford, Epoch) racontent-elles la même histoire ?
3. Les signataires du code publient-ils davantage depuis août 2025 que les non-signataires ?

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import FIGURES
from style import COULEURS_PAYS, ORDRE_PAYS, appliquer_style
from nettoyage import charger_dernier, preparer_epoch, eclater_pays
appliquer_style()
FIGURES.mkdir(exist_ok=True)
PAYS = [p for p in ORDRE_PAYS if p != "Autres"]
from nettoyage import cle_organisation, statut_signature, INFOS_PUBLIABLES

## 0. Chargement

In [ ]:
fmti = charger_dernier("fmti_scores")
signataires = charger_dernier("signataires_code")
epoch = preparer_epoch(charger_dernier("epoch_notable_models"))
print(fmti.groupby("edition")["developpeur"].nunique())
print(f"{len(signataires)} signataires, dont partiels :", signataires.loc[signataires['signature'] != 'complète', 'organisation'].tolist())

## 1. Le score de transparence de Stanford

Score = part des indicateurs satisfaits. **La grille a changé en 2025** : une baisse entre 2024 et 2025 peut venir d'indicateurs plus exigeants, pas seulement d'un recul réel. On compare donc surtout les développeurs **entre eux** au sein d'une même édition, et on interprète l'évolution avec prudence.

In [ ]:
score = (fmti.assign(cle=fmti["developpeur"].map(cle_organisation).fillna(fmti["developpeur"]))
         .groupby(["cle", "edition"])["satisfait"].mean().unstack())
score["statut"] = [statut_signature(c, signataires) for c in score.index]
score.sort_values(score.columns[-2], ascending=False).style.format(
    {c: "{:.0%}" for c in score.columns if c != "statut"}, na_rep="—")

In [ ]:
editions = [c for c in score.columns if c != "statut"]
present = score.dropna(subset=editions[-1:]).sort_values(editions[-1])

fig, ax = plt.subplots(figsize=(9, 5.5))
couleurs_ed = ["#b4b2ab", "#eda100", "#2a78d6"]
for ed, c in zip(editions, couleurs_ed):
    ax.scatter(present[ed], present.index, color=c, s=50, label=ed, zorder=3, edgecolor="white")
for nom, ligne in present[editions].iterrows():
    v = ligne.dropna()
    ax.plot(v.values, [nom] * len(v), color="#d6d5d0", lw=1.5, zorder=1)
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
ax.set_xlabel("Part des indicateurs de transparence satisfaits")
ax.set_title("Transparence des développeurs selon Stanford, par édition")
ax.legend(title="Édition", loc="lower right")
ax.grid(axis="y", visible=False)
fig.tight_layout(); fig.savefig(FIGURES / "30_fmti.png")

> **À interpréter.** Quels développeurs sont les plus transparents en 2025 ? Où se situe Mistral ? Les développeurs ouverts (Meta, DeepSeek, Alibaba…) sont-ils plus transparents que les fermés ? Relier au résultat « ouvert n'est pas transparent » du notebook 1.

## 2. Les deux mesures concordent-elles ?

Pour chaque développeur présent dans les deux sources : score Stanford 2025 contre score de transparence moyen de ses modèles récents dans Epoch (de 0 à 5 informations publiées). On utilise une **corrélation de rang (Spearman)**, adaptée à un très petit nombre d'observations et robuste aux valeurs extrêmes.

In [ ]:
epoch["cle"] = epoch["Organization"].map(cle_organisation)
epoch_recent = (epoch.loc[epoch["annee"] >= 2024].dropna(subset=["cle"])
                .groupby("cle")["score_transparence"].agg(["mean", "count"]))
validation = score[[editions[-1]]].join(epoch_recent, how="inner").dropna()
validation.columns = ["stanford_2025", "epoch_score_moyen", "modeles_epoch"]

rho = validation["stanford_2025"].corr(validation["epoch_score_moyen"], method="spearman")
print(f"{len(validation)} développeurs communs — corrélation de Spearman : {rho:.2f}")
validation.sort_values("stanford_2025", ascending=False).round(2)

> **À interpréter.** Avec si peu de développeurs, une corrélation n'est pas une preuve. Mais va-t-elle dans le sens attendu ? Si les deux mesures divergent pour un développeur, chercher pourquoi (Stanford note aussi l'usage et la gouvernance, Epoch seulement les caractéristiques techniques).

## 3. Avant / après l'AI Act, signataires contre non-signataires

On compare, pour les modèles publiés depuis janvier 2024, la part qui publie sa puissance de calcul et ses données d'entraînement, avant et après le 2 août 2025, selon que le développeur a signé le code européen.

C'est une **comparaison descriptive** du type « différence de différences », pas une estimation causale : peu de modèles après août 2025, des signataires qui n'ont pas été tirés au hasard, et un AI Act qui concerne en principe tous les fournisseurs actifs dans l'UE, signataires ou non.

In [ ]:
recent = epoch.loc[epoch["date"] >= "2024-01-01"].copy()
recent["statut"] = recent["cle"].map(lambda c: statut_signature(c, signataires))
recent["groupe"] = np.where(recent["statut"].str.startswith("complète"), "signataire",
                   np.where(recent["statut"] == "non signataire", "non signataire", "autre"))
recent = recent.loc[recent["groupe"] != "autre"]
recent["periode"] = np.where(recent["apres_ai_act"], "après août 2025", "avant août 2025")

tableau = (recent.groupby(["groupe", "periode"])
           .agg(modeles=("Model", "count"), publie_calcul=("publie_calcul", "mean"),
                publie_donnees=("publie_donnees", "mean"), score=("score_transparence", "mean"))
           .reindex(pd.MultiIndex.from_product([["signataire", "non signataire"],
                                                ["avant août 2025", "après août 2025"]])))
tableau.style.format({"publie_calcul": "{:.0%}", "publie_donnees": "{:.0%}", "score": "{:.2f}"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, var, titre in zip(axes, ["publie_calcul", "publie_donnees"],
                          ["Puissance de calcul publiée", "Taille des données publiée"]):
    for g, c in [("signataire", "#2a78d6"), ("non signataire", "#eb6834")]:
        v = tableau.loc[g, var]
        ax.plot([0, 1], v.values, marker="o", markersize=8, color=c, label=g)
        for xi, yi, n in zip([0, 1], v.values, tableau.loc[g, "modeles"]):
            ax.annotate(f"n={n:.0f}", (xi, yi), textcoords="offset points", xytext=(8, -4),
                        fontsize=8, color="#52514e")
    ax.set_xticks([0, 1], ["avant août 2025", "après août 2025"])
    ax.set_xlim(-0.3, 1.4)
    ax.yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
    ax.set_title(titre)
axes[0].legend()
fig.suptitle("Part des modèles publiant l'information, selon la signature du code", fontweight="bold")
fig.tight_layout(); fig.savefig(FIGURES / "31_ai_act.png")

> **À interpréter.** Les signataires progressent-ils davantage que les non-signataires ? Regarder les effectifs `n` avant de conclure. Discuter au moins deux explications autres qu'un effet de l'AI Act.

## Limites

- Stanford ne couvre qu'une quinzaine de développeurs, et sa grille a changé en 2025.
- Le rapprochement des noms d'organisations entre sources (`cle_organisation` dans `src/nettoyage.py`) est manuel : les organisations non répertoriées sont exclues de la partie 3.
- La liste des signataires est celle publiée par la Commission à la date de collecte ; des signatures peuvent être ajoutées après.